# Notebook 5: Dual-Channel Simultaneous XADC Acquisition Test

This notebook verifies the **Simultaneous Dual-Channel XADC architecture** on the PYNQ-Z2 (`v1.3.0-rc1`).

### Hardware Wiring Check:
* **AD3 W1 (Yellow)** $\rightarrow$ **PYNQ-Z2 A0** (Channel 1 / `Vaux1`)
* **AD3 W2 (Yellow/White)** $\rightarrow$ **PYNQ-Z2 A1** (Channel 2 / `Vaux9`)
* **AD3 GND (Black)** $\rightarrow$ **PYNQ-Z2 GND**

## 1. System Setup & Permission Check

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
import matplotlib.pyplot as plt
import numpy as np
import time

# Ensure USB permissions for AD3 communication
check_usb_permissions()

## 2. Load the Dual-Channel Hardware Overlay
Instantiating `OscilloscopeOverlay()` automatically fetches `v1.3.0-rc1` containing the simultaneous dual-ADC streaming engine.

In [ ]:
ol = OscilloscopeOverlay()
print("✅ Dual-Channel Oscilloscope Overlay loaded successfully!")
print("Active Trigger Configuration:", ol.trigger)

## 3. Generate Dual Test Tones with AD3 Wavegen
We start two distinct test signals:
* **W1 $\rightarrow$ A0:** $1\,\text{kHz}$ Sine wave ($1.0\,\text{V}$ Amplitude, $1.65\,\text{V}$ DC Offset)
* **W2 $\rightarrow$ A1:** $5\,\text{kHz}$ Square wave ($1.0\,\text{V}$ Amplitude, $1.65\,\text{V}$ DC Offset)

In [ ]:
# Start Dual-Channel Signal Generation
ol.wavegen.start(
    shape="Sine", frequency=1000.0, amplitude=1.0, offset=1.65,        # Channel 1 (W1 -> A0)
    ch2_shape="Square", ch2_frequency=5000.0, ch2_amplitude=1.0, ch2_offset=1.65, # Channel 2 (W2 -> A1)
    enable_ch2=True
)
time.sleep(1.0)
print("✅ AD3 active: W1 (1 kHz Sine) -> A0, W2 (5 kHz Square) -> A1")

## 4. Configure Hardware Trigger
Configure the FPGA trigger unit to trigger on the rising edge of Channel 1 (A0) at $1.65\,\text{V}$.

In [ ]:
ol.trigger.configure(mode="Auto", edge="Rising", threshold_volts=1.65, timeout_ms=50.0)
print(f"Hardware Trigger Threshold: {ol.trigger.get_threshold():.2f} V")

## 5. Simultaneous Dual-Channel DMA Capture
Capture both channels synchronously via `ol.capture_stereo()`.

In [ ]:
# Capture 1024 simultaneous sample pairs (1.024 ms window)
v_ch1, v_ch2 = ol.capture_stereo()

print(f"Captured {len(v_ch1)} sample pairs simultaneously!")
print(f"  • CH1 (A0 - Sine)   : Min = {v_ch1.min():.2f} V, Max = {v_ch1.max():.2f} V, Vpp = {v_ch1.max()-v_ch1.min():.2f} V")
print(f"  • CH2 (A1 - Square) : Min = {v_ch2.min():.2f} V, Max = {v_ch2.max():.2f} V, Vpp = {v_ch2.max()-v_ch2.min():.2f} V")

## 6. Waveform Visualization
Plot both channels side-by-side to verify time-alignment and signal fidelity.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, dpi=100)

time_us = np.arange(len(v_ch1))

# Channel 1 (A0)
ax1.plot(time_us[:500], v_ch1[:500], color="#00FFCC", linewidth=1.8, label="CH1: A0 (1 kHz Sine)")
ax1.axhline(1.65, color="#FFA500", linestyle="--", alpha=0.7, label="Trigger (1.65V)")
ax1.set_ylabel("Voltage (V)", fontsize=10)
ax1.set_ylim(0.0, 3.3)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")
ax1.set_title("Dual-Channel Simultaneous Acquisition (1 MSPS)", fontsize=12, fontweight="bold")

# Channel 2 (A1)
ax2.plot(time_us[:500], v_ch2[:500], color="#FF007F", linewidth=1.8, label="CH2: A1 (5 kHz Square)")
ax2.set_xlabel("Time (Microseconds @ 1 MSPS)", fontsize=10)
ax2.set_ylabel("Voltage (V)", fontsize=10)
ax2.set_ylim(0.0, 3.3)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 7. Clean Hardware Shutdown

In [ ]:
ol.close()
print("🔒 AD3 Wavegen stopped and overlay handles cleanly closed.")